# NeuroXVocal — Training & Evaluation
**Server:** Linux | **GPU:** RTX 6000 24 GB | **CUDA:** 12.0

Sections:
- **Part A** — Environment setup
- **Part B** — Data extraction
- **Part C** — Preprocessing
- **Part D** — Training
- **Part E** — Evaluation & VRAM monitor

## Part A — Environment Setup

### A1. Check GPU & CUDA

In [ ]:
import torch
import subprocess

print('PyTorch version  :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())
print('CUDA version     :', torch.version.cuda)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU              : {props.name}')
    print(f'Total VRAM       : {props.total_memory / 1024**3:.1f} GB')
    print(f'CUDA capability  : {props.major}.{props.minor}')
else:
    print('WARNING: No GPU detected!')

# Show nvidia-smi
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('\n', result.stdout)

### A2. Install Dependencies

In [ ]:
# PyTorch with CUDA 12.1 (closest stable release to CUDA 12.0)
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR:', result.stderr[-500:])
    else:
        print('OK:', cmd[:60])

run(f'{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q')
run(f'{sys.executable} -m pip install setuptools wheel -q')
run(f'{sys.executable} -m pip install openai-whisper -q')
run(f'{sys.executable} -m pip install librosa praat-parselmouth soundfile scipy -q')
run(f'{sys.executable} -m pip install transformers==4.45.2 sentencepiece -q')
run(f'{sys.executable} -m pip install scikit-learn pandas numpy tqdm joblib matplotlib -q')

print('\nAll dependencies installed.')

### A3. Clone / Update Repository

In [ ]:
import os, subprocess

REPO_DIR = os.path.expanduser('~/NeuroXVocal')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/homiyano/NeuroXVocal.git', REPO_DIR], check=True)
    print('Cloned to', REPO_DIR)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=True)
    print('Pulled latest changes')

os.listdir(REPO_DIR)

### A4. Configure Paths

In [ ]:
import os

# ── CONFIGURE THESE ─────────────────────────────────────────────────
REPO_DIR       = os.path.expanduser('~/NeuroXVocal')
ADRESSO_ROOT   = os.path.expanduser('~/data/ADReSSo')       # where you placed the dataset
WORK_DIR       = os.path.expanduser('~/data/processed')     # extracted features go here
RESULTS_DIR    = os.path.expanduser('~/NeuroXVocal/results') # models + logs saved here
# ────────────────────────────────────────────────────────────────────

AD_WAV_DIR   = os.path.join(ADRESSO_ROOT, 'diagnosis-train/diagnosis/train/audio/ad')
CN_WAV_DIR   = os.path.join(ADRESSO_ROOT, 'diagnosis-train/diagnosis/train/audio/cn')
TEST_WAV_DIR = os.path.join(ADRESSO_ROOT, 'diagnosis-test/diagnosis/test-dist/audio')

AD_WORK = os.path.join(WORK_DIR, 'ad')
CN_WORK = os.path.join(WORK_DIR, 'cn')

os.makedirs(AD_WORK,     exist_ok=True)
os.makedirs(CN_WORK,     exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify dataset paths
for label, path in [('AD train', AD_WAV_DIR), ('CN train', CN_WAV_DIR), ('Test', TEST_WAV_DIR)]:
    exists = os.path.isdir(path)
    wavs   = len([f for f in os.listdir(path) if f.endswith('.wav')]) if exists else 0
    print(f'{label:<10}: {"OK" if exists else "NOT FOUND"}  ({wavs} .wav files)  {path}')

## Part B — Data Extraction

### B1. Transcribe Audio (Whisper)

In [ ]:
import whisper, time
from pathlib import Path

def transcribe_dir(wav_dir, out_dir, model):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    wavs  = sorted(Path(wav_dir).glob('*.wav'))
    print(f'  Found {len(wavs)} files')
    times = []
    for wav in wavs:
        txt_file = Path(out_dir) / (wav.stem + '.txt')
        if txt_file.exists():
            continue
        t0      = time.perf_counter()
        result  = model.transcribe(str(wav), fp16=True)
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
        txt_file.write_text(result['text'], encoding='utf-8')
        print(f'  {wav.name:<35}  {elapsed:.2f}s')
    if times:
        total = sum(times)
        print(f'  ── Total: {total:.1f}s  |  Avg: {total/len(times):.2f}s  |  Files: {len(times)}')
    else:
        print('  All files already transcribed — skipped.')

whisper_model = whisper.load_model('base')

print('Transcribing AD...')
t = time.perf_counter()
transcribe_dir(AD_WAV_DIR, AD_WORK, whisper_model)
print(f'AD done: {time.perf_counter()-t:.1f}s\n')

print('Transcribing CN...')
t = time.perf_counter()
transcribe_dir(CN_WAV_DIR, CN_WORK, whisper_model)
print(f'CN done: {time.perf_counter()-t:.1f}s')

### B2. Extract Audio Features (librosa + parselmouth)

In [ ]:
import time, sys

AD_FEATURES_RAW = os.path.join(AD_WORK, 'audio_features_ad.csv')
CN_FEATURES_RAW = os.path.join(CN_WORK, 'audio_features_cn.csv')
EXTRACT_FEAT    = os.path.join(REPO_DIR, 'src/data_extraction/extract_audio_features.py')

print('Extracting AD audio features...')
t0 = time.perf_counter()
!{sys.executable} {EXTRACT_FEAT} {AD_WAV_DIR} --output_csv {AD_FEATURES_RAW}
print(f'AD features time: {time.perf_counter()-t0:.1f}s\n')

print('Extracting CN audio features...')
t0 = time.perf_counter()
!{sys.executable} {EXTRACT_FEAT} {CN_WAV_DIR} --output_csv {CN_FEATURES_RAW}
print(f'CN features time: {time.perf_counter()-t0:.1f}s')

### B3. Extract Audio Embeddings (Wav2Vec2)

In [ ]:
import time, sys

AD_EMB_RAW   = os.path.join(AD_WORK, 'audio_embeddings_ad.csv')
CN_EMB_RAW   = os.path.join(CN_WORK, 'audio_embeddings_cn.csv')
EXTRACT_EMB  = os.path.join(REPO_DIR, 'src/data_extraction/extract_audio_embeddings.py')

print('Extracting AD audio embeddings (Wav2Vec2)...')
t0 = time.perf_counter()
!{sys.executable} {EXTRACT_EMB} {AD_WAV_DIR} --output_csv {AD_EMB_RAW}
print(f'AD embeddings time: {time.perf_counter()-t0:.1f}s\n')

print('Extracting CN audio embeddings (Wav2Vec2)...')
t0 = time.perf_counter()
!{sys.executable} {EXTRACT_EMB} {CN_WAV_DIR} --output_csv {CN_EMB_RAW}
print(f'CN embeddings time: {time.perf_counter()-t0:.1f}s')

## Part C — Preprocessing

### C1. Preprocess Transcriptions

In [ ]:
import sys

AD_TEXT_PROC = os.path.join(WORK_DIR, 'ad_text_processed')
CN_TEXT_PROC = os.path.join(WORK_DIR, 'cn_text_processed')
PREPROC_TEXT = os.path.join(REPO_DIR, 'src/data_processing/preprocess_texts.py')

!{sys.executable} {PREPROC_TEXT} {AD_WORK} {AD_TEXT_PROC}
!{sys.executable} {PREPROC_TEXT} {CN_WORK} {CN_TEXT_PROC}
print('Text preprocessing done.')

### C2. Preprocess Audio Features

In [ ]:
import shutil, sys

SCALER_FEAT      = os.path.join(REPO_DIR, 'src/inference/scaler_params_audio_features.pkl')
PREPROC_FEAT     = os.path.join(REPO_DIR, 'src/data_processing/preprocess_audio_features.py')
AD_FEAT_PROC_DIR = os.path.join(WORK_DIR, 'ad_features_processed')
CN_FEAT_PROC_DIR = os.path.join(WORK_DIR, 'cn_features_processed')
os.makedirs(AD_FEAT_PROC_DIR, exist_ok=True)
os.makedirs(CN_FEAT_PROC_DIR, exist_ok=True)

!{sys.executable} {PREPROC_FEAT} --input_path {AD_FEATURES_RAW} --output_path {AD_FEAT_PROC_DIR} --scaler_path {SCALER_FEAT}
!{sys.executable} {PREPROC_FEAT} --input_path {CN_FEATURES_RAW} --output_path {CN_FEAT_PROC_DIR} --scaler_path {SCALER_FEAT}

AD_FEAT_FINAL = os.path.join(AD_WORK, 'audio_features_ad_processed.csv')
CN_FEAT_FINAL = os.path.join(CN_WORK, 'audio_features_cn_processed.csv')
shutil.copy(os.path.join(AD_FEAT_PROC_DIR, 'audio_features_ad.csv'), AD_FEAT_FINAL)
shutil.copy(os.path.join(CN_FEAT_PROC_DIR, 'audio_features_cn.csv'), CN_FEAT_FINAL)
print('Audio features preprocessed.')

### C3. Preprocess Audio Embeddings

In [ ]:
import pandas as pd, numpy as np, joblib

SCALER_EMB = os.path.join(REPO_DIR, 'src/inference/scaler_params_audio_emb.pkl')

def preprocess_embeddings(input_csv, output_csv, scaler_path):
    df          = pd.read_csv(input_csv)
    patient_ids = df['patient_id'].values
    features    = df.drop(columns=['patient_id'])
    features    = features.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler      = joblib.load(scaler_path)
    scaled      = scaler.transform(features)
    df_out      = pd.DataFrame(scaled, columns=features.columns)
    df_out.insert(0, 'patient_id', patient_ids)
    df_out.to_csv(output_csv, index=False)
    print(f'Saved {output_csv}')

AD_EMB_FINAL = os.path.join(AD_WORK, 'audio_embeddings_ad_processed.csv')
CN_EMB_FINAL = os.path.join(CN_WORK, 'audio_embeddings_cn_processed.csv')

preprocess_embeddings(AD_EMB_RAW, AD_EMB_FINAL, SCALER_EMB)
preprocess_embeddings(CN_EMB_RAW, CN_EMB_FINAL, SCALER_EMB)

### C4. Verify All Files Ready

In [ ]:
from pathlib import Path
import pandas as pd

checks = [
    ('AD features',   AD_FEAT_FINAL),
    ('CN features',   CN_FEAT_FINAL),
    ('AD embeddings', AD_EMB_FINAL),
    ('CN embeddings', CN_EMB_FINAL),
]
for name, path in checks:
    df = pd.read_csv(path)
    print(f'{name:<16}: {df.shape[0]} patients  {df.shape[1]-1} features')

ad_feat_ids = set(pd.read_csv(AD_FEAT_FINAL)['patient_id'].astype(str))
cn_feat_ids = set(pd.read_csv(CN_FEAT_FINAL)['patient_id'].astype(str))
ad_txt_ids  = {p.stem for p in Path(AD_TEXT_PROC).glob('*.txt')}
cn_txt_ids  = {p.stem for p in Path(CN_TEXT_PROC).glob('*.txt')}

ad_miss = ad_feat_ids - ad_txt_ids
cn_miss = cn_feat_ids - cn_txt_ids
if ad_miss: print(f'WARNING: {len(ad_miss)} AD patients missing transcripts:', ad_miss)
if cn_miss: print(f'WARNING: {len(cn_miss)} CN patients missing transcripts:', cn_miss)
if not ad_miss and not cn_miss:
    print('\nAll patients have transcripts. Ready to train!')

## Part D — Training

### D1. Write Config & Start Training

In [ ]:
config = f'''import os

AD_TEXT_DIR        = r"{AD_TEXT_PROC}"
CN_TEXT_DIR        = r"{CN_TEXT_PROC}"
AD_CSV             = r"{AD_FEAT_FINAL}"
CN_CSV             = r"{CN_FEAT_FINAL}"
AD_EMBEDDING_CSV   = r"{AD_EMB_FINAL}"
CN_EMBEDDING_CSV   = r"{CN_EMB_FINAL}"

TEXT_EMBEDDING_MODEL   = 'microsoft/deberta-v3-base'
NUM_MFCC_FEATURES      = 47
NUM_EMBEDDING_FEATURES = 768
AUDIO_CHANNELS         = 1
CUDA                   = True

BATCH_SIZE              = 8
EPOCHS                  = 200
LEARNING_RATE           = 1e-5
WEIGHT_DECAY            = 1e-4
NUM_FOLDS               = 5
SAVE_BEST_MODEL         = True
EARLY_STOPPING_PATIENCE = 10

SAVE_MODEL_PATH = r"{RESULTS_DIR}/model"
LOG_PATH        = r"{RESULTS_DIR}/training.log"
'''

config_path = os.path.join(REPO_DIR, 'src/train/config.py')
with open(config_path, 'w') as f:
    f.write(config)
print('Config written.')
print(config)

In [ ]:
import sys, time

MAIN_PY = os.path.join(REPO_DIR, 'src/train/main.py')

print('Starting training...')
print(f'Results will be saved to: {RESULTS_DIR}')
print('─' * 60)

t0 = time.perf_counter()
!cd {os.path.join(REPO_DIR, 'src/train')} && {sys.executable} main.py
total = time.perf_counter() - t0

print('─' * 60)
print(f'Total training time: {total/3600:.2f} hours  ({total/60:.1f} min)')

### D2. Training & Validation Loss Plots

In [ ]:
import re, os
import matplotlib.pyplot as plt

LOG_PATH = os.path.join(RESULTS_DIR, 'training.log')
pattern  = re.compile(r'Fold (\d+), Epoch (\d+), Train Loss: ([\d.]+), Val Loss: ([\d.]+)')

folds = {}
with open(LOG_PATH) as f:
    for line in f:
        m = pattern.search(line)
        if m:
            fold, epoch = int(m.group(1)), int(m.group(2))
            if fold not in folds:
                folds[fold] = {'epochs': [], 'train': [], 'val': []}
            folds[fold]['epochs'].append(epoch)
            folds[fold]['train'].append(float(m.group(3)))
            folds[fold]['val'].append(float(m.group(4)))

num_folds = len(folds)
print(f'Parsed {num_folds} folds')
for fold, data in sorted(folds.items()):
    best_epoch = data['epochs'][data['val'].index(min(data['val']))]
    print(f'  Fold {fold}: {len(data["epochs"])} epochs | best val loss = {min(data["val"]):.4f} at epoch {best_epoch}')

# ── Per-fold subplots ────────────────────────────────────────────────
cols = min(num_folds, 3)
rows = (num_folds + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
fig.suptitle('NeuroXVocal — Training vs Validation Loss per Fold', fontsize=14, fontweight='bold')
axes_flat = axes.flatten() if num_folds > 1 else [axes]

for idx, (fold, data) in enumerate(sorted(folds.items())):
    ax         = axes_flat[idx]
    best_idx   = data['val'].index(min(data['val']))
    best_epoch = data['epochs'][best_idx]
    best_val   = data['val'][best_idx]

    ax.plot(data['epochs'], data['train'], color='#2196F3', linewidth=2, label='Train loss')
    ax.plot(data['epochs'], data['val'],   color='#F44336', linewidth=2, label='Val loss')
    ax.axvline(x=best_epoch, color='#4CAF50', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.scatter([best_epoch], [best_val], color='#4CAF50', s=80, zorder=5)
    ax.annotate(f'best\nepoch {best_epoch}\n{best_val:.4f}',
                xy=(best_epoch, best_val),
                xytext=(best_epoch+0.5, best_val+(max(data['val'])-min(data['val']))*0.15),
                fontsize=8, color='#2E7D32',
                arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.2))
    ax.set_title(f'Fold {fold}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=1)

for idx in range(num_folds, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'loss_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

# ── All-folds overlay ────────────────────────────────────────────────
FOLD_COLORS = ['#E53935','#8E24AA','#1E88E5','#00897B','#F4511E']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('All Folds Overlay', fontsize=13, fontweight='bold')

for idx, (fold, data) in enumerate(sorted(folds.items())):
    color    = FOLD_COLORS[idx % len(FOLD_COLORS)]
    best_idx = data['val'].index(min(data['val']))
    ax1.plot(data['epochs'], data['train'], color=color, linewidth=1.8, label=f'Fold {fold}')
    ax2.plot(data['epochs'], data['val'],   color=color, linewidth=1.8, label=f'Fold {fold}')
    ax2.scatter([data['epochs'][best_idx]], [data['val'][best_idx]], color=color, s=70, zorder=5)

for ax, title in [(ax1,'Training Loss'), (ax2,'Validation Loss')]:
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=1)

plt.tight_layout()
overlay_path = os.path.join(RESULTS_DIR, 'loss_overlay.png')
plt.savefig(overlay_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {overlay_path}')

### D3. Select Best Fold

In [ ]:
import glob

model_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.pth')))
print('Saved checkpoints:')
for m in model_files:
    print(f'  {os.path.basename(m)}  ({os.path.getsize(m)/1e6:.1f} MB)')

# Pick best fold (lowest val loss)
best_fold = min(folds, key=lambda f: min(folds[f]['val']))
best_loss = min(folds[best_fold]['val'])
best_ep   = folds[best_fold]['epochs'][folds[best_fold]['val'].index(best_loss)]

print(f'\nBest fold: Fold {best_fold}  |  val loss = {best_loss:.4f}  |  epoch = {best_ep}')

best_src  = os.path.join(RESULTS_DIR, f'model_fold{best_fold}_best.pth')
best_dst  = os.path.join(RESULTS_DIR, 'best.pth')
import shutil
shutil.copy(best_src, best_dst)
print(f'Copied {os.path.basename(best_src)} → best.pth')

## Part E — Evaluation & VRAM Monitor

### E1. VRAM Monitor Class

In [ ]:
import time, torch

class VRAMMonitor:
    def __init__(self, device):
        self.device       = device
        self.dtype        = device.type
        self._global_peak = 0.0
        self.load_records = []

    def current_mb(self):
        return torch.cuda.memory_allocated(self.device) / 1024**2 if self.dtype == 'cuda' else 0.0

    def peak_mb(self):
        return torch.cuda.max_memory_allocated(self.device) / 1024**2 if self.dtype == 'cuda' else 0.0

    def reset_peak(self):
        if self.dtype == 'cuda': torch.cuda.reset_peak_memory_stats(self.device)

    def sync(self):
        if self.dtype == 'cuda': torch.cuda.synchronize(self.device)

    def measure(self, label):
        return _Stage(self, label)

    def record_load(self, name, fn):
        self.sync()
        before = self.current_mb()
        obj    = fn()
        self.sync()
        after  = self.current_mb()
        delta  = after - before
        self.load_records.append({'model': name, 'before_mb': round(before,1),
                                   'after_mb': round(after,1), 'delta_mb': round(delta,1)})
        print(f'  Loaded {name:<26}  before={before:6.1f} MB  after={after:6.1f} MB  delta=+{delta:.1f} MB')
        return obj

    def update_global_peak(self, v):
        if v > self._global_peak: self._global_peak = v

    def global_peak_mb(self):
        return self._global_peak


class _Stage:
    def __init__(self, mon, label):
        self.mon   = mon
        self.label = label
        self.vram_before = self.vram_after = self.peak = self.elapsed = 0.0

    def __enter__(self):
        self.mon.sync()
        self.mon.reset_peak()
        self.vram_before = self.mon.current_mb()
        self._t0         = time.perf_counter()
        return self

    def __exit__(self, *_):
        self.mon.sync()
        self.elapsed    = time.perf_counter() - self._t0
        self.vram_after = self.mon.current_mb()
        self.peak       = self.mon.peak_mb()
        self.mon.update_global_peak(self.peak)

print('VRAMMonitor ready.')

### E2. Load All Models (with VRAM tracking)

In [ ]:
import sys, re, tempfile, shutil, subprocess, joblib, warnings
import whisper, soundfile as sf
import pandas as pd, numpy as np
from pathlib import Path
from math import gcd
from scipy.signal import resample_poly
from transformers import AutoTokenizer, Wav2Vec2Model, Wav2Vec2Processor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

sys.path.insert(0, os.path.join(REPO_DIR, 'src/train'))
from models import NeuroXVocal

SCALER_FEAT = os.path.join(REPO_DIR, 'src/inference/scaler_params_audio_features.pkl')
SCALER_EMB  = os.path.join(REPO_DIR, 'src/inference/scaler_params_audio_emb.pkl')
DROP_COLS   = ['jitter_local','shimmer_local','formant_1_mean','formant_1_std',
               'formant_2_mean','formant_2_std','formant_3_mean','formant_3_std','class']
TEXT_MODEL  = 'microsoft/deberta-v3-base'
MODEL_PATH  = os.path.join(RESULTS_DIR, 'best.pth')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mon    = VRAMMonitor(device)
print(f'Device: {device}\n')

wmodel    = mon.record_load('Whisper base',
                lambda: whisper.load_model('base'))
tokenizer = mon.record_load('DeBERTa tokenizer',
                lambda: AutoTokenizer.from_pretrained(TEXT_MODEL))
processor = mon.record_load('Wav2Vec2 processor',
                lambda: Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h'))
emodel    = mon.record_load('Wav2Vec2 model',
                lambda: Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').to(device).eval())

def _load_clf():
    clf = NeuroXVocal(num_audio_features=47, num_embedding_features=768, text_embedding_model=TEXT_MODEL)
    sd  = torch.load(MODEL_PATH, map_location=device)
    if 'module.' in list(sd.keys())[0]:
        from collections import OrderedDict
        sd = OrderedDict((k.replace('module.',''), v) for k, v in sd.items())
    clf.load_state_dict(sd)
    return clf.to(device).eval()

clf = mon.record_load('NeuroXVocal (DeBERTa)', _load_clf)

print('\n── Model Load Summary ──────────────────────────────────────────')
print(f'  {"Model":<28}  {"Before":>9}  {"After":>9}  {"Delta":>9}')
print('  ' + '-'*58)
for r in mon.load_records:
    print(f'  {r["model"]:<28}  {r["before_mb"]:>8.1f}M  {r["after_mb"]:>8.1f}M  +{r["delta_mb"]:>7.1f}M')

### E3. Run Evaluation with VRAM & Timing Monitor
3 pipelines tracked per patient: Whisper → Wav2Vec2 → NeuroXVocal classifier

In [ ]:
from tqdm import tqdm

def transcribe(wav_path):
    r = wmodel.transcribe(str(wav_path), fp16=torch.cuda.is_available())
    t = r['text'].lower()
    return re.sub(r'[^a-zA-Z0-9\s.,!?]', '', ' '.join(t.split()))

def get_audio_features(wav_path):
    script = os.path.join(REPO_DIR, 'src/data_extraction/extract_audio_features.py')
    tmp    = Path(tempfile.mkdtemp())
    csv    = tmp / 'f.csv'
    subprocess.run([sys.executable, script, str(wav_path.parent),
                    '--output_csv', str(csv)], capture_output=True)
    df  = pd.read_csv(csv)
    shutil.rmtree(tmp)
    row = df[df['patient_id'] == wav_path.stem]
    pid = row['patient_id'].values
    row = row.drop(columns=['patient_id'] + DROP_COLS, errors='ignore')
    row = row.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaled = joblib.load(SCALER_FEAT).transform(row)
    out = pd.DataFrame(scaled, columns=row.columns)
    out.insert(0, 'patient_id', pid)
    return out

def get_embeddings(wav_path):
    speech, sr = sf.read(str(wav_path), always_2d=True)
    speech     = speech.mean(axis=1).astype(np.float32)
    if sr != 16000:
        g = gcd(sr, 16000)
        speech = resample_poly(speech, 16000//g, sr//g).astype(np.float32)
    inp = processor(speech, sampling_rate=16000, return_tensors='pt', padding=True)
    inp = {k: v.to(device) for k, v in inp.items()}
    with torch.no_grad():
        emb = emodel(**inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return joblib.load(SCALER_EMB).transform(emb.reshape(1,-1)).flatten()

def predict(text, feat_row, emb):
    tok  = tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')
    tok  = {k: v.to(device) for k, v in tok.items()}
    af   = torch.tensor(feat_row.drop(columns=['patient_id']).iloc[0].values.astype(float),
                        dtype=torch.float32).unsqueeze(0).to(device)
    embt = torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        conf = torch.sigmoid(clf(tok, af, embt)).item()
    return (1 if conf > 0.5 else 0), conf

# ── Main loop ────────────────────────────────────────────────────────
wav_files       = sorted(Path(TEST_WAV_DIR).glob('*.wav'))
results         = []
monitor_records = []

for wav in tqdm(wav_files, desc='Patients'):
    pid = wav.stem
    try:
        with mon.measure('1. Whisper transcription') as s1:
            text = transcribe(wav)
        monitor_records.append({'patient': pid, 'pipeline': '1. Whisper transcription',
                                'vram_before_mb': round(s1.vram_before,1), 'vram_peak_mb': round(s1.peak,1),
                                'vram_after_mb': round(s1.vram_after,1), 'time_sec': round(s1.elapsed,3)})

        t0        = time.perf_counter()
        feat_proc = get_audio_features(wav)
        monitor_records.append({'patient': pid, 'pipeline': '2a. AudioFeat (subprocess)',
                                'vram_before_mb': None, 'vram_peak_mb': None,
                                'vram_after_mb': None, 'time_sec': round(time.perf_counter()-t0,3)})

        with mon.measure('2b. Wav2Vec2 embedding') as s2:
            emb_proc = get_embeddings(wav)
        monitor_records.append({'patient': pid, 'pipeline': '2b. Wav2Vec2 embedding',
                                'vram_before_mb': round(s2.vram_before,1), 'vram_peak_mb': round(s2.peak,1),
                                'vram_after_mb': round(s2.vram_after,1), 'time_sec': round(s2.elapsed,3)})

        with mon.measure('3. NeuroXVocal classify') as s3:
            pred, conf = predict(text, feat_proc, emb_proc)
        monitor_records.append({'patient': pid, 'pipeline': '3. NeuroXVocal classify',
                                'vram_before_mb': round(s3.vram_before,1), 'vram_peak_mb': round(s3.peak,1),
                                'vram_after_mb': round(s3.vram_after,1), 'time_sec': round(s3.elapsed,3)})

        results.append({'patient_id': pid, 'prediction': pred,
                        'label': 'AD' if pred==1 else 'CN',
                        'confidence': round(conf,4), 'transcription': text[:100]})
    except Exception as e:
        print(f'ERROR {pid}: {e}')
        results.append({'patient_id': pid, 'prediction': None,
                        'label': 'ERROR', 'confidence': None, 'transcription': str(e)})

OUTPUT_CSV  = os.path.join(RESULTS_DIR, 'predictions.csv')
MONITOR_CSV = os.path.join(RESULTS_DIR, 'vram_monitor.csv')

df_results = pd.DataFrame(results)
df_monitor = pd.DataFrame(monitor_records)
df_results.to_csv(OUTPUT_CSV,  index=False)
df_monitor.to_csv(MONITOR_CSV, index=False)
print(f'Predictions saved → {OUTPUT_CSV}')
print(f'VRAM log saved    → {MONITOR_CSV}')
df_results[['patient_id','label','confidence']]

### E4. Prediction Summary & VRAM Report

In [ ]:
valid = df_results.dropna(subset=['prediction'])
print(f'Total processed : {len(valid)} / {len(df_results)}')
print(f'Predicted AD    : {(valid["prediction"]==1).sum()}')
print(f'Predicted CN    : {(valid["prediction"]==0).sum()}')
print(f'\nConfidence stats:')
print(valid['confidence'].describe().round(4))

print('\n')

# Model load table
df_load = pd.DataFrame(mon.load_records)
print('MODEL LOAD — VRAM COST')
print('=' * 62)
print(df_load.to_string(index=False))

# Per-pipeline summary
tracked = df_monitor[df_monitor['vram_peak_mb'].notna()].copy()
summary = tracked.groupby('pipeline').agg(
    peak_max  = ('vram_peak_mb', 'max'),
    peak_min  = ('vram_peak_mb', 'min'),
    peak_mean = ('vram_peak_mb', 'mean'),
    time_mean = ('time_sec',     'mean'),
).round(2)

print('\nVRAM & TIMING SUMMARY')
print('=' * 72)
print(f'{"Pipeline":<28}  {"Peak max":>9}  {"Peak min":>9}  {"Peak avg":>9}  {"Avg time":>9}')
print('-' * 72)
for pipe, row in summary.iterrows():
    print(f'{pipe:<28}  {row.peak_max:>8.1f}M  {row.peak_min:>8.1f}M  {row.peak_mean:>8.1f}M  {row.time_mean:>8.2f}s')
print('=' * 72)
print(f'Global peak VRAM : {mon.global_peak_mb():.1f} MB')
print(f'Global min  VRAM : {tracked["vram_peak_mb"].min():.1f} MB')

### E5. Accuracy vs Ground Truth (optional)

In [ ]:
# Set path to ground truth CSV if available: columns (patient_id, label) where 1=AD 0=CN
LABELS_CSV = None

if LABELS_CSV is not None:
    gt     = pd.read_csv(LABELS_CSV)
    merged = df_results.merge(gt, on='patient_id', suffixes=('_pred','_true'))
    merged = merged.dropna(subset=['prediction'])
    y_true = merged['label_true'].astype(int).values
    y_pred = merged['prediction'].astype(int).values
    acc    = accuracy_score(y_true, y_pred)
    print(f'Test Accuracy: {acc*100:.2f}%  ({int(acc*len(y_true))}/{len(y_true)})\n')
    print(classification_report(y_true, y_pred, target_names=['CN','AD']))
    print('Confusion Matrix:')
    print(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=['True CN','True AD'], columns=['Pred CN','Pred AD']))
else:
    print('LABELS_CSV not set. Provide ground truth CSV to compute accuracy.')